## BASE SQL DANS PYTHON
https://www.sqlite.org/index.html 
SQLite est une lib qui fournit une base de donnée légère qui ne nécessite pas de server et est pratique pour servir de support d'écriture externe à des programmes externes comme python

In [1]:
import sqlite3

In [2]:
# Pour utiliser on ouvre une connection et on cree une DB
conn = sqlite3.connect('example.db')
# You can also supply the special name :memory: to create a database in RAM.

In [3]:
# une fois connectée, on utilise cursor pour déclencher des ordres SQL
c = conn.cursor()

In [6]:
c.execute('''DROP TABLE IF EXISTS actions''')

In [7]:
# SQL Create table
c.execute('''CREATE TABLE IF NOT EXISTS actions
             (date date, trans text, action text, quantite real, prix real)''')

In [8]:
# SQL Insert
c.execute("INSERT INTO actions VALUES ('2020-01-05','BUY','EPSILON',100,35.14)")
c.execute("INSERT INTO actions VALUES ('2020-01-06','SELL','EPSILON',100,42)")

In [9]:
# Sauvegarder les modifications dans la base
# le faire avant de fermer
conn.commit()

In [10]:
# On ferme la connection quand on a fini il We can also close the connection if we are done with it.
# Bien faire le commit sinon les modif n'auront pas été réalisées
conn.close()

In [11]:
# Les données sont persistantes donc on peut repartir
conn = sqlite3.connect('example.db')
c = conn.cursor()

In [12]:
# QUERY
# pour récupérer data après un select, on peut considerére le cursor comme iterable 
# ou appliquer fetchone() qui recupere 1 ligne
# oul fetchall() toutes les lignes
c.execute('SELECT * FROM actions WHERE action="EPSILON"')
#print(c.fetchone())
print(c.fetchall())

[('2020-01-05', 'BUY', 'EPSILON', 100.0, 35.14), ('2020-01-06', 'SELL', 'EPSILON', 100.0, 42.0)]


In [13]:
# Larger example that inserts many records at a time
mouvements = [('2020-03-28', 'BUY', 'IBM', 1000, 45.00),
             ('2020-04-05', 'BUY', 'MSFT', 1000, 72.00),
              ('2020-04-05', 'BUY', 'MSFT', 1000, 72.00),
              ('2020-05-05', 'BUY', 'MSFT', 1000, 74.00),
              ('2020-06-05', 'BUY', 'MSFT', 1000, 61.00),
              ('2020-07-05', 'BUY', 'MSFT', 1000, 55.00),
             ('2020-04-06', 'SELL', 'IBM', 500, 53.00),
              ('2020-01-05','BUY','EPSILON',200,38),
              ('2020-02-05','SELL','EPSILON',300,35.14),
              ('2020-03-05','BUY','EPSILON',100,36),
              ('2020-04-05','BUY','EPSILON',500,37)
            ]
c.executemany('INSERT INTO actions VALUES (?,?,?,?,?)', mouvements)

In [14]:
for row in c.execute('SELECT * FROM actions ORDER BY prix'):
        print(row)

('2020-01-05', 'BUY', 'EPSILON', 100.0, 35.14)
('2020-02-05', 'SELL', 'EPSILON', 300.0, 35.14)
('2020-03-05', 'BUY', 'EPSILON', 100.0, 36.0)
('2020-04-05', 'BUY', 'EPSILON', 500.0, 37.0)
('2020-01-05', 'BUY', 'EPSILON', 200.0, 38.0)
('2020-01-06', 'SELL', 'EPSILON', 100.0, 42.0)
('2020-03-28', 'BUY', 'IBM', 1000.0, 45.0)
('2020-04-06', 'SELL', 'IBM', 500.0, 53.0)
('2020-07-05', 'BUY', 'MSFT', 1000.0, 55.0)
('2020-06-05', 'BUY', 'MSFT', 1000.0, 61.0)
('2020-04-05', 'BUY', 'MSFT', 1000.0, 72.0)
('2020-04-05', 'BUY', 'MSFT', 1000.0, 72.0)
('2020-05-05', 'BUY', 'MSFT', 1000.0, 74.0)


In [15]:
# WINDOW FUNCTION
# la fonction rank permet de créer un rang pour chaque ligne correspondant à une modalité selon un ordre
# ici on cree un rang par ordre decroissant de date pour chaque action

In [16]:
c.execute('select *,RANK() OVER (PARTITION BY action ORDER BY date desc) as dern from actions')
print(c.fetchall())

[('2020-04-05', 'BUY', 'EPSILON', 500.0, 37.0, 1), ('2020-03-05', 'BUY', 'EPSILON', 100.0, 36.0, 2), ('2020-02-05', 'SELL', 'EPSILON', 300.0, 35.14, 3), ('2020-01-06', 'SELL', 'EPSILON', 100.0, 42.0, 4), ('2020-01-05', 'BUY', 'EPSILON', 100.0, 35.14, 5), ('2020-01-05', 'BUY', 'EPSILON', 200.0, 38.0, 5), ('2020-04-06', 'SELL', 'IBM', 500.0, 53.0, 1), ('2020-03-28', 'BUY', 'IBM', 1000.0, 45.0, 2), ('2020-07-05', 'BUY', 'MSFT', 1000.0, 55.0, 1), ('2020-06-05', 'BUY', 'MSFT', 1000.0, 61.0, 2), ('2020-05-05', 'BUY', 'MSFT', 1000.0, 74.0, 3), ('2020-04-05', 'BUY', 'MSFT', 1000.0, 72.0, 4), ('2020-04-05', 'BUY', 'MSFT', 1000.0, 72.0, 4)]


In [17]:
# si on souhaite afficher la première date de transaction en face de chaque transaction
c.execute('select *,FIRST_VALUE(date) OVER (PARTITION BY action ORDER BY date desc) as date_der from actions')
print(c.fetchall())

[('2020-04-05', 'BUY', 'EPSILON', 500.0, 37.0, '2020-04-05'), ('2020-03-05', 'BUY', 'EPSILON', 100.0, 36.0, '2020-04-05'), ('2020-02-05', 'SELL', 'EPSILON', 300.0, 35.14, '2020-04-05'), ('2020-01-06', 'SELL', 'EPSILON', 100.0, 42.0, '2020-04-05'), ('2020-01-05', 'BUY', 'EPSILON', 100.0, 35.14, '2020-04-05'), ('2020-01-05', 'BUY', 'EPSILON', 200.0, 38.0, '2020-04-05'), ('2020-04-06', 'SELL', 'IBM', 500.0, 53.0, '2020-04-06'), ('2020-03-28', 'BUY', 'IBM', 1000.0, 45.0, '2020-04-06'), ('2020-07-05', 'BUY', 'MSFT', 1000.0, 55.0, '2020-07-05'), ('2020-06-05', 'BUY', 'MSFT', 1000.0, 61.0, '2020-07-05'), ('2020-05-05', 'BUY', 'MSFT', 1000.0, 74.0, '2020-07-05'), ('2020-04-05', 'BUY', 'MSFT', 1000.0, 72.0, '2020-07-05'), ('2020-04-05', 'BUY', 'MSFT', 1000.0, 72.0, '2020-07-05')]


In [19]:
# ou la dernière date
#LAST_VALUE(Name) OVER (
#        PARTITION BY AlbumId
#        ORDER BY Bytes DESC
#    ) AS LargestTrack